In [1]:
import pandas as pd
import anthropic
import duckdb

In [2]:
client = anthropic.Anthropic()

In [3]:
# Load batch ID
with open('/app/data/sentiment_batch_id.txt', 'r') as f:
    batch_id = f.read().strip()

In [5]:
def parse_sentiment(text):
    text = text.lower().strip()
    if text.startswith('positive'):
        return 'positive'
    elif text.startswith('negative'):
        return 'negative'
    elif text.startswith('neutral'):
        return 'neutral'
    else:
        # Try to find the word anywhere in the response
        for word in ['positive', 'negative', 'neutral']:
            if word in text:
                return word
        return 'unknown'

In [6]:
# Check status
batch = client.messages.batches.retrieve(batch_id)
print(f"Status: {batch.processing_status}")
print(f"Request counts: {batch.request_counts}")

if batch.processing_status == 'ended':
    # Retrieve results
    results = []
    for entry in client.messages.batches.results(batch_id):
        if entry.result.type == 'succeeded':
            batch_index = int(entry.custom_id.replace('sentiment_', ''))
            sentiment = parse_sentiment(entry.result.message.content[0].text)
            results.append({
                'batch_index': batch_index,
                'claude_sentiment': sentiment
            })

    results_df = pd.DataFrame(results)
    print(f"Results retrieved: {len(results_df):,}")

    # Join back with mapping to get review_id
    mapping_df = pd.read_csv('/app/data/sentiment_batch_mapping.csv')
    final_df = results_df.merge(mapping_df, on='batch_index')
    final_df = final_df[['review_id', 'claude_sentiment']]

    print(f"Final results: {len(final_df):,}")
    print(final_df['claude_sentiment'].value_counts())

    # Save to DuckDB
    conn = duckdb.connect('/app/data/analytics.duckdb')
    conn.execute("CREATE SCHEMA IF NOT EXISTS llm_outputs")
    conn.execute("DROP TABLE IF EXISTS llm_outputs.review_sentiments")
    conn.execute("CREATE TABLE llm_outputs.review_sentiments AS SELECT * FROM final_df")
    print("Saved to llm_outputs.review_sentiments in DuckDB")

    # Backup to CSV
    final_df.to_csv('/app/data/review_sentiments.csv', index=False)
    print("Backup saved to /app/data/review_sentiments.csv")

else:
    print(f"Batch not ready yet. Status: {batch.processing_status}")
    print(f"Request counts: {batch.request_counts}")

Status: ended
Request counts: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=41940)
Results retrieved: 41,940
Final results: 41,940
claude_sentiment
positive    25468
negative    13520
neutral      2830
unknown       122
Name: count, dtype: int64
Saved to llm_outputs.review_sentiments in DuckDB
Backup saved to /app/data/review_sentiments.csv
